# SER Ablation Study — Colab Runner

| Variant | Config | run_name |
|---|---|---|
| Baseline | crnn_ravdess.yaml | ravdess_baseline |
| +Harmonic | harmonic_only_ravdess.yaml | ravdess_harmonic |
| +FreqPos | freqpos_only_ravdess.yaml | ravdess_freqpos |
| +Both | harmonic_freqpos.yaml | ravdess_harmonic_freqpos |

**Your friend runs Baseline. You run the other three.**

In [ ]:
# ── Cell 1: clone repo & install deps ─────────────────────────────────────
import os

REPO_URL = "https://github.com/rushankgoyal/crnn-ser.git"
REPO_DIR = "/content/crnn-ser"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!pip install -q -r requirements.txt

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("No GPU — training will be slow but will still run.")

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/cs231n-ser"
os.makedirs(f"{DRIVE_ROOT}/data/ravdess", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)
print("Drive mounted. DRIVE_ROOT =", DRIVE_ROOT)

In [ ]:
# ── Cell 4: Ensure preprocessed data exists (auto-downloads if needed) ────
DATA_ROOT = f"{DRIVE_ROOT}/data/ravdess"
os.makedirs(DATA_ROOT, exist_ok=True)

missing = [s for s in ["train", "val", "test"]
           if not os.path.exists(f"{DATA_ROOT}/{s}.npz")]

if not missing:
    print("All splits found:")
    for s in ["train", "val", "test"]:
        size = os.path.getsize(f"{DATA_ROOT}/{s}.npz") / 1e6
        print(f"  {s}.npz  {size:.1f} MB")
else:
    print(f"Missing splits {missing} — downloading and preprocessing RAVDESS (~5 min)...")

    RAW_DIR = "/content/ravdess_raw"
    ZIP_PATH = "/content/ravdess.zip"
    ZIP_URL = "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip"
    os.makedirs(RAW_DIR, exist_ok=True)

    if not os.path.exists(ZIP_PATH):
        !wget -q --show-progress -O {ZIP_PATH} {ZIP_URL}
    !unzip -q {ZIP_PATH} -d {RAW_DIR}

    actor_dirs = [d for d in os.listdir(RAW_DIR) if d.startswith("Actor_")]
    print(f"Extracted {len(actor_dirs)} actor folders.")

    !python data/preprocess.py --dataset ravdess --raw_dir {RAW_DIR} --out_dir {DATA_ROOT}

    !rm -rf {RAW_DIR} {ZIP_PATH}

    # Verify
    still_missing = [s for s in ["train", "val", "test"]
                     if not os.path.exists(f"{DATA_ROOT}/{s}.npz")]
    if still_missing:
        raise RuntimeError(f"Preprocessing failed — still missing: {still_missing}")
    print("Preprocessing complete:")
    for s in ["train", "val", "test"]:
        size = os.path.getsize(f"{DATA_ROOT}/{s}.npz") / 1e6
        print(f"  {s}.npz  {size:.1f} MB  ✓")

In [ ]:
# ── Cell 5: Patch configs with runtime data_root ───────────────────────────
import yaml

VARIANTS = [
    ("crnn_ravdess.yaml",            "ravdess_baseline"),
    ("harmonic_only_ravdess.yaml",   "ravdess_harmonic"),
    ("freqpos_only_ravdess.yaml",    "ravdess_freqpos"),
    ("harmonic_freqpos.yaml",        "ravdess_harmonic_freqpos"),
]

RUNTIME_CONFIGS = {}
os.makedirs("/content/runtime_configs", exist_ok=True)

for cfg_file, run_name in VARIANTS:
    with open(f"configs/{cfg_file}") as f:
        cfg = yaml.safe_load(f)
    cfg["data_root"] = DATA_ROOT
    cfg["run_name"] = run_name
    out_path = f"/content/runtime_configs/{run_name}.yaml"
    with open(out_path, "w") as f:
        yaml.dump(cfg, f)
    RUNTIME_CONFIGS[run_name] = out_path
    print(f"  {run_name}: data_root={DATA_ROOT}")

In [ ]:
# ── Cell 6: CPU unit tests ─────────────────────────────────────────────────
!python tests/test_components.py

In [ ]:
# ── Cell 7: GPU smoke train — 2 epochs, confirm loss drops ────────────────
import subprocess, sys

smoke_cfg_path = "/content/runtime_configs/ravdess_harmonic_freqpos.yaml"
with open(smoke_cfg_path) as f:
    scfg = yaml.safe_load(f)
scfg["train"]["epochs"] = 2
smoke_out = "/content/runtime_configs/smoke.yaml"
with open(smoke_out, "w") as f:
    yaml.dump(scfg, f)

result = subprocess.run([sys.executable, "train.py", "--config", smoke_out])
if result.returncode != 0:
    raise RuntimeError(f"Smoke train FAILED (exit {result.returncode}) — fix the error above before proceeding.")
print("\nSmoke train OK — GPU path confirmed.")

## Full Ablation Training

Run Cells 8b / 8c / 8d — these are your three variants.  
**Skip 8a** — that's your friend's baseline.

Each run saves `runs/{run_name}/best.pt`.

In [ ]:
# ── Cell 8a: Baseline — your friend runs this, not you ────────────────────
# !python train.py --config {RUNTIME_CONFIGS['ravdess_baseline']}

In [ ]:
# ── Cell 8b: +Harmonic only ───────────────────────────────────────────────
!python train.py --config {RUNTIME_CONFIGS['ravdess_harmonic']}

In [ ]:
# ── Cell 8c: +FreqPos only ────────────────────────────────────────────────
!python train.py --config {RUNTIME_CONFIGS['ravdess_freqpos']}

In [ ]:
# ── Cell 8d: +Both ────────────────────────────────────────────────────────
!python train.py --config {RUNTIME_CONFIGS['ravdess_harmonic_freqpos']}

In [ ]:
# ── Cell 9: Save checkpoints to Drive ─────────────────────────────────────
import shutil
for run_name in ["ravdess_harmonic", "ravdess_freqpos", "ravdess_harmonic_freqpos"]:
    src = f"runs/{run_name}"
    dst = f"{DRIVE_ROOT}/runs/{run_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  Saved runs/{run_name} → Drive")
    else:
        print(f"  Skipped {run_name} (not trained yet)")

## Evaluation

**Before running Cell 10**, get your friend's baseline checkpoint from Drive and copy it in:
```python
os.makedirs("runs/ravdess_baseline", exist_ok=True)
!cp "{DRIVE_ROOT}/runs/ravdess_baseline/best.pt" runs/ravdess_baseline/best.pt
```
Then run Cell 10 — it skips variants whose checkpoint is missing, so it's safe to run partially.

In [ ]:
# ── Cell 10: Evaluate all trained variants ─────────────────────────────────
for run_name, cfg_path in RUNTIME_CONFIGS.items():
    ckpt = f"runs/{run_name}/best.pt"
    if not os.path.exists(ckpt):
        print(f"  SKIP {run_name}: no checkpoint")
        continue
    print(f"\n{'='*60}\nEvaluating: {run_name}\n{'='*60}")
    !python evaluate.py --config {cfg_path} --checkpoint {ckpt}

In [ ]:
# ── Cell 11: Sync results to Drive ────────────────────────────────────────
for run_name in RUNTIME_CONFIGS:
    src = f"results/{run_name}"
    dst = f"{DRIVE_ROOT}/results/{run_name}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"  Saved results/{run_name} → Drive")

## Comparison Plots

To include your friend's baseline:
1. They copy their `results/ravdess_baseline/` to Drive
2. You pull it in: `shutil.copytree(f"{DRIVE_ROOT}/results/ravdess_baseline", "results/ravdess_baseline", dirs_exist_ok=True)`
3. Run Cell 12 — it picks up all four variants automatically

In [ ]:
# ── Cell 12: Latency-accuracy comparison + UAR bar chart ──────────────────
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

HOP_MS = 10.0

VARIANTS_PLOT = [
    ("Baseline",  "ravdess_baseline",            "#2c3e50", "-"),
    ("+Harmonic", "ravdess_harmonic",             "#e74c3c", "--"),
    ("+FreqPos",  "ravdess_freqpos",              "#3498db", "-."),
    ("+Both",     "ravdess_harmonic_freqpos",      "#27ae60", ":"),
]

loaded = {}
for label, run_name, color, ls in VARIANTS_PLOT:
    path = f"results/{run_name}/metrics.json"
    if os.path.exists(path):
        with open(path) as f:
            loaded[label] = (json.load(f), color, ls)
    else:
        print(f"  Missing {path} — skipping {label}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle("Ablation: Harmonic Dilation + Frequency Positional Encoding",
             fontsize=14, fontweight="bold", y=1.02)

ax = axes[0]
for label, (metrics, color, ls) in loaded.items():
    curve = np.array(metrics["latency_curve"])
    t_ms  = np.arange(len(curve)) * HOP_MS
    ax.plot(t_ms, curve, color=color, linestyle=ls, linewidth=2.2,
            label=f"{label}  (UAR={metrics['uar']:.3f})")
ax.axvline(500, color="grey", linestyle=":", linewidth=1, alpha=0.6)
ax.text(510, 0.03, "500 ms", color="grey", fontsize=9)
ax.set_xlabel("Elapsed time (ms)", fontsize=12)
ax.set_ylabel("Frame accuracy", fontsize=12)
ax.set_title("Latency–Accuracy Curve", fontsize=13)
ax.legend(fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.25)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

ax2 = axes[1]
names  = list(loaded.keys())
uars   = [loaded[n][0]["uar"] for n in names]
colors = [loaded[n][1] for n in names]
bars = ax2.bar(names, uars, color=colors, alpha=0.85,
               edgecolor="white", linewidth=1.5, width=0.55)
for bar, u in zip(bars, uars):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.008, f"{u:.3f}",
             ha="center", va="bottom", fontsize=11, fontweight="bold")
ax2.set_ylabel("UAR", fontsize=12)
ax2.set_title("Final UAR by Variant", fontsize=13)
ax2.set_ylim(0, 1.0)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.grid(True, alpha=0.25, axis="y")

plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/ablation_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved → results/ablation_comparison.png")

In [ ]:
# ── Cell 13: Confusion matrix grid ────────────────────────────────────────
EMOTIONS = ["happy", "sad", "angry", "neutral"]
n = len(loaded)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5))
if n == 1:
    axes = [axes]

for ax, (label, (metrics, color, _)) in zip(axes, loaded.items()):
    cm = np.array(metrics["confusion_matrix"], dtype=float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(EMOTIONS, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(EMOTIONS, fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{label}\nUAR={metrics['uar']:.3f}", fontsize=11)
    for i in range(4):
        for j in range(4):
            v = cm_norm[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v > 0.55 else "black", fontsize=9)

fig.suptitle("Confusion Matrices (row-normalized)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("results/confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved → results/confusion_matrices.png")

In [ ]:
# ── Cell 14: Save everything to Drive ─────────────────────────────────────
import shutil
for fname in ["ablation_comparison.png", "confusion_matrices.png"]:
    src = f"results/{fname}"
    if os.path.exists(src):
        shutil.copy(src, f"{DRIVE_ROOT}/results/{fname}")
        print(f"  Saved {fname} → Drive")